# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayush0121n/flyrank-ml-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb)

**Lane:** CTR / Engagement Opportunity Scoring  
**Author:** Ayush Narkhede  
**Input:** Validated client-holdout model (GB ROC-AUC 0.781, base rate 22.6%) — see `work/outputs/capstone_metrics.json` and `w06_validation_audit.ipynb`.

This notebook turns model scores into a **human-reviewed action queue** with reason codes, limits, and no-go rules. Language is decision-support only.


## 1. Ranked actions + reason codes

**Archetype → action mapping** (from model score + dominant features):

| Tier | Score band (approx) | Archetype | Action | Typical reason codes |
|------|---------------------|-----------|--------|----------------------|
| 1 | Top ~10–15% | High opportunity | Prioritize title + meta rewrite; check intent match | `LOW_CTR_MAY`, `SOLID_IMPR_MAY`, `MID_POS` |
| 2 | Next ~20% | Medium opportunity | Content freshness / depth review; optional FAQ | `MOD_CTR`, `HIGH_IMPR`, `THIN_CONTENT` |
| 3 | Remainder of ranked queue | Monitor / protect | Keep in normal refresh cadence; watch drops | `STABLE_CTR`, `LOW_VOLUME` |

Reason codes are derived from the features the model actually used (prior-month CTR, impressions, position, structural length) — not from June label fields.


In [ ]:
import json
from pathlib import Path
import pandas as pd

# Receipt metrics from the sealed evaluation
metrics_path = Path("work/outputs/capstone_metrics.json")
if not metrics_path.exists():
    metrics_path = Path("../outputs/capstone_metrics.json")
m = json.loads(metrics_path.read_text())

# Illustrative ranked queue (structure the paper will reuse).
# In production this would be scored content rows; here we show the action schema.
rows = [
    {"rank": 1, "tier": 1, "action": "Prioritize title + meta rewrite", "reason_codes": "LOW_CTR_MAY|SOLID_IMPR_MAY|MID_POS",
     "rationale": "Low prior CTR with meaningful impressions — first place editors look."},
    {"rank": 2, "tier": 1, "action": "Prioritize title + meta rewrite", "reason_codes": "LOW_CTR_MAY|HIGH_IMPR_MAY",
     "rationale": "High visibility, weak capture historically — review snippet fit."},
    {"rank": 3, "tier": 2, "action": "Content freshness / depth review", "reason_codes": "MOD_CTR|THIN_CONTENT|SOLID_IMPR_MAY",
     "rationale": "Moderate CTR + short content — possible depth or FAQ gap."},
    {"rank": 4, "tier": 2, "action": "Content freshness / depth review", "reason_codes": "AVG_POS_HIGH|MOD_CTR",
     "rationale": "Weaker average position with middling CTR — monitor + light edit."},
    {"rank": 5, "tier": 3, "action": "Monitor / protect", "reason_codes": "STABLE_CTR|LOW_VOLUME",
     "rationale": "Low volume or stable CTR — do not burn editorial time first."},
]

queue = pd.DataFrame(rows)
print("=== Ranked action queue (schema for paper) ===")
print(queue.to_string(index=False))
print()
print("Model context: GB AUC={:.3f}, baseline AUC={:.3f}, test base rate={:.1%}".format(
    m["gb_auc"], m["baseline_auc"], m["label_rate_test"]))
print("Top features:", ", ".join(list(m["top_features"].keys())[:5]))

# Hold in memory for export cell
ACTION_QUEUE = queue
METRICS = m


=== Ranked action queue (schema for paper) ===
 rank  tier                           action                        reason_codes                                                             rationale
    1     1  Prioritize title + meta rewrite  LOW_CTR_MAY|SOLID_IMPR_MAY|MID_POS Low prior CTR with meaningful impressions — first place editors look.
    2     1  Prioritize title + meta rewrite           LOW_CTR_MAY|HIGH_IMPR_MAY      High visibility, weak capture historically — review snippet fit.
    3     2 Content freshness / depth review MOD_CTR|THIN_CONTENT|SOLID_IMPR_MAY             Moderate CTR + short content — possible depth or FAQ gap.
    4     2 Content freshness / depth review                AVG_POS_HIGH|MOD_CTR     Weaker average position with middling CTR — monitor + light edit.
    5     3                Monitor / protect               STABLE_CTR|LOW_VOLUME          Low volume or stable CTR — do not burn editorial time first.

Model context: GB AUC=0.781, baseline AUC=0.57

## 2. Intended use and limits

**Intended users:** editorial / SEO operators who can change titles, meta, or body content for pseudonymized content items in this warehouse panel.

**Intended use:** rank a limited review queue so scarce editorial time goes to pages that **look** like CTR opportunity in this data — measured association under a client-holdout, not a guarantee of traffic lift.

**Limits (honest):**
- Observed on May→June 2026 windows only; other months may differ.
- No experimental design; external SERP/competitor shocks are unobserved.
- AI-referral sessions were deliberately kept out of the primary label (sparse).
- Score is a ranking signal for humans, not an auto-publish switch.

**Decay / refresh insight:** Prior-month CTR dominates importance. Queues should be **rebuilt when a new complete month lands**, not treated as permanent truth.


In [ ]:
intended_use = {
    "users": ["editorial", "SEO operators"],
    "decision": "Which visible pages to review for metadata/content this cycle?",
    "valid_when": "Same feature recipe (prior month + static dim) and client-holdout style evaluation still holds",
    "invalid_when": [
        "Treating scores as causal traffic forecasts",
        "Auto-rewriting or auto-publishing without human review",
        "Applying the queue to clients/windows never represented in training",
    ],
    "refresh_cadence": "Rebuild after each complete month of GSC data",
}
print(json.dumps(intended_use, indent=2))


{
  "users": [
    "editorial",
    "SEO operators"
  ],
  "decision": "Which visible pages to review for metadata/content this cycle?",
  "valid_when": "Same feature recipe (prior month + static dim) and client-holdout style evaluation still holds",
  "invalid_when": [
    "Treating scores as causal traffic forecasts",
    "Auto-rewriting or auto-publishing without human review",
    "Applying the queue to clients/windows never represented in training"
  ],
  "refresh_cadence": "Rebuild after each complete month of GSC data"
}


## 3. Human review + the no-go list

**Before acting on a Tier-1/2 item, a human checks:**
1. Intent still matches the query set the page is ranking for.
2. Title/meta are not already optimized (avoid thrashing).
3. No brand / legal / medical claims risk in the proposed edit.
4. Page is still indexed and not intentionally thin (category hubs, etc.).

### No-go — what should NOT be automated
- Sending emails, posting comments, or publishing CMS changes from the model score alone.
- Deleting or merging pages solely from a low score.
- Using the score as a performance review metric for writers.
- Claiming “this rewrite will increase clicks by X%.”

The agent pattern from FL-07 applies here too: **propose, never auto-act.**


In [ ]:
human_review = {
    "checkpoints": [
        "intent_match",
        "already_optimized_check",
        "brand_legal_risk",
        "index_status",
    ],
    "no_go_automation": [
        "auto_publish",
        "auto_delete_or_merge",
        "writer_performance_scoring",
        "causal_traffic_promises",
    ],
}
print("Human review checkpoints:", ", ".join(human_review["checkpoints"]))
print("No-go automation:")
for x in human_review["no_go_automation"]:
    print("  -", x)


Human review checkpoints: intent_match, already_optimized_check, brand_legal_risk, index_status
No-go automation:
  - auto_publish
  - auto_delete_or_merge
  - writer_performance_scoring
  - causal_traffic_promises


## 4. Monitoring / retrain triggers

Rebuild or re-evaluate the model when any of these fire:

| Trigger | Why |
|---------|-----|
| New complete GSC month available | Feature/label windows move; queue goes stale |
| Client mix shifts materially | Holdout assumptions change |
| Base rate of “opportunity” moves > ~5 pp | Thresholds and Precision@K interpretation drift |
| Rule baseline closes most of the AUC gap | Model may no longer add decision-support value |
| Editorial feedback: top-K systematically wrong intent | Feature set or label definition needs revision |

**Cost / value (practical, non-production):**  
Scoring is cheap (tabular sklearn). Editorial time is the scarce resource — that is why the playbook is a **short ranked queue**, not a full-site rewrite list.


In [ ]:
monitoring = {
    "rebuild_when": [
        "new_complete_gsc_month",
        "client_mix_shift",
        "opportunity_base_rate_shift_gt_5pp",
        "baseline_closes_auc_gap",
        "editorial_top_k_intent_failures",
    ],
    "scarce_resource": "editorial_hours",
    "model_cost": "low_tabular_sklearn",
}
print(json.dumps(monitoring, indent=2))


{
  "rebuild_when": [
    "new_complete_gsc_month",
    "client_mix_shift",
    "opportunity_base_rate_shift_gt_5pp",
    "baseline_closes_auc_gap",
    "editorial_top_k_intent_failures"
  ],
  "scarce_resource": "editorial_hours",
  "model_cost": "low_tabular_sklearn"
}


## 5. Exports for the paper

Write the action schema and a compact playbook JSON under `work/outputs/`.  
(Queue CSVs with real content hashes stay out of git by design; this notebook documents the schema the paper’s recommendations section uses.)


In [ ]:
import os
from pathlib import Path

out = Path("work/outputs")
if not out.exists():
    out = Path("../outputs")
out.mkdir(parents=True, exist_ok=True)

# Schema-level queue for the paper (no client-identifying IDs)
queue_path = out / "action_playbook_queue.csv"
ACTION_QUEUE.to_csv(queue_path, index=False)

playbook = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "model": "GradientBoosting",
    "metrics": {
        "gb_auc": METRICS["gb_auc"],
        "baseline_auc": METRICS["baseline_auc"],
        "label_rate_test": METRICS["label_rate_test"],
    },
    "tiers": {
        "1": "Prioritize title + meta rewrite",
        "2": "Content freshness / depth review",
        "3": "Monitor / protect",
    },
    "reason_code_vocab": [
        "LOW_CTR_MAY", "SOLID_IMPR_MAY", "HIGH_IMPR_MAY", "MID_POS",
        "AVG_POS_HIGH", "MOD_CTR", "THIN_CONTENT", "STABLE_CTR", "LOW_VOLUME",
    ],
    "no_go": [
        "auto_publish", "auto_delete", "causal_traffic_promises",
    ],
    "language": "observed / directional / decision-support",
}
playbook_path = out / "action_playbook.json"
playbook_path.write_text(json.dumps(playbook, indent=2))

print("Wrote", queue_path)
print("Wrote", playbook_path)
print(playbook_path.read_text()[:500], "...")


Wrote work/outputs/action_playbook_queue.csv
Wrote work/outputs/action_playbook.json
{
  "lane": "CTR / Engagement Opportunity Scoring",
  "model": "GradientBoosting",
  "metrics": {
    "gb_auc": 0.7811166440272592,
    "baseline_auc": 0.5765846540089778,
    "label_rate_test": 0.22595126122274475
  },
  "tiers": {
    "1": "Prioritize title + meta rewrite",
    "2": "Content freshness / depth review",
    "3": "Monitor / protect"
  },
  "reason_code_vocab": [
    "LOW_CTR_MAY",
    "SOLID_IMPR_MAY",
    "HIGH_IMPR_MAY",
    "MID_POS",
    "AVG_POS_HIGH",
    "MOD_CTR",
    "TH ...


## Self-check

- [x] Ranked actions + reason codes tied to the CTR opportunity model  
- [x] Intended use and limits in public-safe language  
- [x] Human review checkpoints + explicit no-go automation list  
- [x] Monitoring / retrain triggers + cost/value note  
- [x] Exports written under `work/outputs/` for the paper  
- [x] No client names, domains, or private queries  
- [x] Notebook runs top to bottom  

**Repo path:** `work/notebooks/w07_action_playbook.ipynb`
